# Imports

In [4]:
import os

import pandas as pd

In [6]:
os.chdir('..')
print(os.getcwd())

d:\Study\MLDS\recommendation-engine


## 1. Load Raw Data

Loading the three MovieLens 1M source files (`users.dat`, `movies.dat`, `ratings.dat`).
These are `::`-delimited, not comma-separated, so `pandas.read_csv` needs explicit
`sep` and `engine` arguments. Starting with `ratings.dat` since it has no encoding
concerns — `movies.dat` will need a Latin-1 encoding fix, tackled separately below.

In [7]:
ratings = pd.read_csv(
    'data/raw/ml-1m/ratings.dat',
    sep='::',
    engine='python',
    names=['UserID', 'MovieID', 'Rating', 'Timestamp'],
    header=None
)

print(ratings.head())
print(ratings.dtypes)

   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291
UserID       int64
MovieID      int64
Rating       int64
Timestamp    int64
dtype: object


In [10]:
movies = pd.read_csv(
    'data/raw/ml-1m/movies.dat',
    sep='::',
    engine= 'python',
    names=['MovieID', 'Title', 'Genres'],
    header=None,
    encoding='latin-1'
)

print(movies.head())
print(movies.dtypes)

   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy
MovieID    int64
Title        str
Genres       str
dtype: object


In [11]:
users = pd.read_csv(
    'data/raw/ml-1m/users.dat',
    sep='::',
    engine='python',
    names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip-code'],
    header=None
)

print(users.head())
print(users.dtypes)

   UserID Gender  Age  Occupation Zip-code
0       1      F    1          10    48067
1       2      M   56          16    70072
2       3      M   25          15    55117
3       4      M   45           7    02460
4       5      M   25          20    55455
UserID        int64
Gender          str
Age           int64
Occupation    int64
Zip-code        str
dtype: object


### Ingestion Validation

In [12]:
for name, df in [('ratings', ratings), ('movies', movies), ('users', users)]:
    print(f"--- {name} ---")
    print("Shape:", df.shape)
    print("Missing values:\n", df.isnull().sum())
    print("Duplicate rows:", df.duplicated().sum())
    print()

--- ratings ---
Shape: (1000209, 4)
Missing values:
 UserID       0
MovieID      0
Rating       0
Timestamp    0
dtype: int64
Duplicate rows: 0

--- movies ---
Shape: (3883, 3)
Missing values:
 MovieID    0
Title      0
Genres     0
dtype: int64
Duplicate rows: 0

--- users ---
Shape: (6040, 5)
Missing values:
 UserID        0
Gender        0
Age           0
Occupation    0
Zip-code      0
dtype: int64
Duplicate rows: 0



#### Check Ratings Range (If any value is out of Range)
#### Check Gender Values (If any values are there except M/F)

In [20]:
print(ratings['Rating'].unique())
print(users['Gender'].unique())

[5 3 4 2 1]
<StringArray>
['F', 'M']
Length: 2, dtype: str


In [22]:
invalid_ratings = ratings[(ratings['Rating'] < 1) | (ratings['Rating'] > 5)]
print("Invalid ratings count : ", len(invalid_ratings))

invalid_gender = users[~users['Gender'].isin(['M', 'F'])]
print("Invalid gender count : ", len(invalid_gender))

Invalid ratings count :  0
Invalid gender count :  0


#### A more explicit check for duplicated values. If a user rates a movie more than once

In [23]:
duplicate_pairs = ratings.duplicated(subset=['UserID', 'MovieID'])
print("Duplicate (UserID, MovieID) pairs:", duplicate_pairs.sum())

Duplicate (UserID, MovieID) pairs: 0


## Data Cleaning — Summary

Validation confirmed the raw MovieLens 1M data required no cleaning:
- No missing values across all three files
- No duplicate rows
- All ratings within valid 1-5 range
- Gender values restricted to {M, F}
- No duplicate (UserID, MovieID) pairs

Proceeding directly to feature engineering.